In [1]:
import numpy as np
from math import erf, sqrt, pi

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Product


# ----------------------------
# Normal pdf/cdf (no scipy)
# ----------------------------
def norm_pdf(z: float) -> float:
    return np.exp(-0.5 * z * z) / sqrt(2.0 * pi)

def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))


# ----------------------------
# Extract (variance, lengthscale) from fitted sklearn kernel
# Supports: RBF or ConstantKernel * RBF (Product)
# ----------------------------
def extract_rbf_params(kernel):
    variance = 1.0
    length_scale = None

    if isinstance(kernel, RBF):
        length_scale = kernel.length_scale

    elif isinstance(kernel, Product):
        # ConstantKernel * RBF (either order)
        k1, k2 = kernel.k1, kernel.k2
        if isinstance(k1, ConstantKernel) and isinstance(k2, RBF):
            variance = float(k1.constant_value)
            length_scale = k2.length_scale
        elif isinstance(k2, ConstantKernel) and isinstance(k1, RBF):
            variance = float(k2.constant_value)
            length_scale = k1.length_scale
        else:
            raise ValueError(
                "Unsupported Product kernel. Use ConstantKernel*RBF (and no other factors)."
            )
    else:
        raise ValueError("Unsupported kernel type. Use RBF or ConstantKernel*RBF.")

    length_scale = np.asarray(length_scale, dtype=float)
    return variance, length_scale


# ----------------------------
# k(x, X) and ∂k/∂x for RBF
# ----------------------------
def rbf_k_and_grad(x, X, variance, length_scale):
    """
    x: (d,)
    X: (n,d)
    variance: scalar (sigma_f^2)
    length_scale: (d,) or scalar (broadcasted)
    returns:
      kx  : (n,)
      dkx : (n,d) where dkx[i,j] = ∂k(x,X[i]) / ∂x_j
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    X = np.asarray(X, dtype=float)
    d = X.shape[1]

    if length_scale.ndim == 0:
        ls = np.full(d, float(length_scale))
    elif length_scale.size == 1:
        ls = np.full(d, float(length_scale.ravel()[0]))
    else:
        ls = length_scale.reshape(-1)
        if ls.size != d:
            raise ValueError(f"length_scale has size {ls.size}, expected {d}.")

    diff = (x[None, :] - X)                 # (n,d)
    scaled = diff / ls[None, :]             # (n,d)
    sqdist = np.sum(scaled**2, axis=1)      # (n,)
    kx = variance * np.exp(-0.5 * sqdist)   # (n,)

    # ∂/∂x_j k = k * (-(x_j - X_ij) / ls_j^2)
    dkx = kx[:, None] * (-(diff) / (ls[None, :] ** 2))  # (n,d)
    return kx, dkx


# ----------------------------
# EI (minimization) and gradient
# ----------------------------
def ei_min_and_grad(gp: GaussianProcessRegressor, x, f_best, eps_sigma=1e-12):
    """
    Minimization EI:
      EI(x) = (f_best - mu)*Phi(z) + sigma*phi(z),
      z = (f_best - mu)/sigma

    Gradient simplification:
      ∇EI = -∇mu * Phi(z) + ∇sigma * phi(z)

    Assumes gp.kernel_ is RBF or ConstantKernel*RBF, and noise via gp.alpha.
    """
    X = gp.X_train_
    alpha = gp.alpha_          # (n,)
    L = gp.L_                  # Cholesky of (K + alpha_noise I)
    variance, length_scale = extract_rbf_params(gp.kernel_)

    kx, dkx = rbf_k_and_grad(x, X, variance, length_scale)  # (n,), (n,d)

    # Posterior mean: mu = k(x,X) K^{-1} y = kx^T alpha
    mu = float(kx @ alpha)
    grad_mu = dkx.T @ alpha  # (d,)

    # Solve beta = (K + alpha_noise I)^{-1} kx using Cholesky factors
    v = np.linalg.solve(L, kx)
    beta = np.linalg.solve(L.T, v)

    # Posterior variance of latent f: sigma^2 = k(x,x) - kx^T beta
    # For RBF (with ConstantKernel scale), k(x,x) = variance
    sigma2 = float(variance - (kx @ beta))
    sigma2 = max(sigma2, 0.0)
    sigma = sqrt(sigma2)

    if sigma < eps_sigma:
        return 0.0, np.zeros_like(grad_mu)

    # grad sigma^2 = -2 * dkx^T beta   (since k(x,x) is constant in x for RBF)
    grad_sigma2 = -2.0 * (dkx.T @ beta)  # (d,)
    grad_sigma = 0.5 * grad_sigma2 / max(sigma, eps_sigma)

    diff = f_best - mu
    z = diff / sigma
    Phi = norm_cdf(z)
    phi = norm_pdf(z)

    ei = diff * Phi + sigma * phi
    grad_ei = (-grad_mu) * Phi + grad_sigma * phi
    return float(ei), grad_ei


# ----------------------------
# Multi-start projected gradient ascent on EI
# ----------------------------
def maximize_ei(
    gp,
    bounds,                 # (2,d)
    f_best,
    n_restarts=30,
    n_steps=200,
    step0=0.05,
    backtrack=0.5,
    min_step=1e-6,
    grad_clip=10.0,
    seed=0
):
    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)
    lo, hi = bounds[0], bounds[1]
    d = lo.size

    def project(x):
        return np.minimum(np.maximum(x, lo), hi)

    best_x = None
    best_val = -np.inf

    for _ in range(n_restarts):
        x = lo + (hi - lo) * rng.random(d)
        x = project(x)

        val, _ = ei_min_and_grad(gp, x, f_best)
        step = step0

        for _it in range(n_steps):
            val, g = ei_min_and_grad(gp, x, f_best)

            gn = np.linalg.norm(g)
            if gn < 1e-10:
                break
            if gn > grad_clip:
                g = g * (grad_clip / (gn + 1e-12))

            # backtracking line search to ensure EI increases
            accepted = False
            step_try = step
            while step_try >= min_step:
                x_new = project(x + step_try * g)
                val_new, _ = ei_min_and_grad(gp, x_new, f_best)
                if val_new >= val:
                    x, val = x_new, val_new
                    accepted = True
                    step = step_try / backtrack if backtrack < 1.0 else step_try
                    break
                step_try *= backtrack

            if not accepted:
                break

        if val > best_val:
            best_val = val
            best_x = x.copy()

    return best_x, float(best_val)


In [2]:
# ----------------------------
# Example BO step (minimization)
# ----------------------------

# Toy objective to minimize (replace with your expensive f)
def f(x):
    x = np.asarray(x)
    return (x[0] - 0.2)**2 + (x[1] - 0.8)**2 + 0.1*np.sin(8*x[0])

# Initial data
n0, d = 12, 2
bounds = np.array([[0.0, 0.0], [1.0, 1.0]])

rng = np.random.default_rng(1)
X = rng.random((n0, d))
y = np.array([f(x) for x in X])

# Fit GP: smooth prior (RBF), noise via alpha
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2))
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,          # observation noise variance (set larger if noisy)
    normalize_y=True,    # okay here; EI uses f_best from original y below
    n_restarts_optimizer=5,
    random_state=0
)
gp.fit(X, y)

# Best observed (since minimizing)
f_best = float(np.min(y))

# Maximize EI using gradients
x_next, ei_val = maximize_ei(gp, bounds, f_best, n_restarts=40, n_steps=250, step0=0.03, seed=123)

print("x_next =", x_next)
print("EI(x_next) =", ei_val)

# Evaluate objective and append (one BO iteration)
y_next = f(x_next)
print("f(x_next) =", y_next)

x_next = [0.         0.79928693]
EI(x_next) = 1.482942597881445
f(x_next) = 0.04000050846893191


In [10]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Product


# -------------------------
# True objective (MINIMIZE)
# -------------------------
def f_true(x):
    x = np.asarray(x)
    return (x - 0.35) ** 2 + 0.08 * np.sin(10 * x) + 0.03 * np.cos(22 * x)


# --------------------------------------------
# Extract (variance, lengthscale) from kernel_
# Supports: RBF or ConstantKernel * RBF
# --------------------------------------------
def extract_rbf_params_1d(kernel):
    variance = 1.0
    length_scale = None

    if isinstance(kernel, RBF):
        length_scale = float(np.asarray(kernel.length_scale).ravel()[0])

    elif isinstance(kernel, Product):
        k1, k2 = kernel.k1, kernel.k2
        if isinstance(k1, ConstantKernel) and isinstance(k2, RBF):
            variance = float(k1.constant_value)
            length_scale = float(np.asarray(k2.length_scale).ravel()[0])
        elif isinstance(k2, ConstantKernel) and isinstance(k1, RBF):
            variance = float(k2.constant_value)
            length_scale = float(np.asarray(k1.length_scale).ravel()[0])
        else:
            raise ValueError("Use ConstantKernel * RBF (no extra factors).")
    else:
        raise ValueError("Unsupported kernel. Use RBF or ConstantKernel*RBF.")

    return variance, length_scale


# -------------------------
# RBF k(x,X) and dk/dx (1D)
# -------------------------
def rbf_k_and_dk_1d(xq, Xtrain, variance, length_scale):
    xq = np.asarray(xq).reshape(-1)        # (m,)
    Xtrain = np.asarray(Xtrain).reshape(-1)  # (n,)

    diff = xq[:, None] - Xtrain[None, :]   # (m,n)
    Kqx = variance * np.exp(-0.5 * (diff ** 2) / (length_scale ** 2))
    dKqx = Kqx * (-(diff) / (length_scale ** 2))
    return Kqx, dKqx


# ---------------------------------------------------------
# Posterior mean/std and derivatives mu'(x), sigma'(x) (1D)
# ---------------------------------------------------------
def posterior_and_derivatives_1d(gp, xq, eps_sigma=1e-12):
    xq = np.asarray(xq).reshape(-1)
    Xtrain = gp.X_train_.reshape(-1)     # (n,)
    alpha_vec = gp.alpha_.reshape(-1)    # (n,) = (K + noise I)^-1 y
    L = gp.L_                            # Cholesky of (K + noise I)

    variance, length_scale = extract_rbf_params_1d(gp.kernel_)

    Kqx, dKqx = rbf_k_and_dk_1d(xq, Xtrain, variance, length_scale)  # (m,n)

    mu = Kqx @ alpha_vec
    mu_p = dKqx @ alpha_vec

    # beta = (K + noise I)^-1 k(x,X)^T for all xq (multiple RHS)
    RHS = Kqx.T  # (n,m)
    v = np.linalg.solve(L, RHS)
    beta = np.linalg.solve(L.T, v)  # (n,m)

    # Latent variance: sigma^2 = k(x,x) - k^T beta ; for RBF, k(x,x) = variance
    sigma2 = variance - np.sum(Kqx * beta.T, axis=1)
    sigma2 = np.maximum(sigma2, 0.0)
    sigma = np.sqrt(sigma2)

    # sigma2' = -2 * dk^T beta
    sigma2_p = -2.0 * np.sum(dKqx * beta.T, axis=1)
    sigma_safe = np.maximum(sigma, eps_sigma)
    sig_p = 0.5 * sigma2_p / sigma_safe
    sig_p[sigma < eps_sigma] = 0.0

    return mu, sigma, mu_p, sig_p


# -----------------------------------------
# EI (minimization) and derivative dEI/dx
# EI = (f_best - mu) Phi(z) + sigma phi(z)
# z = (f_best - mu)/sigma
# dEI/dx = -(dmu/dx) Phi(z) + (dsigma/dx) phi(z)
# -----------------------------------------
def ei_min_and_derivative_1d(gp, xq, f_best, eps_sigma=1e-12):
    mu, sigma, mu_p, sig_p = posterior_and_derivatives_1d(gp, xq, eps_sigma=eps_sigma)
    sigma_safe = np.maximum(sigma, eps_sigma)

    diff = f_best - mu
    z = diff / sigma_safe

    Phi = norm.cdf(z)
    phi = norm.pdf(z)

    EI = diff * Phi + sigma_safe * phi
    dEI = (-mu_p) * Phi + sig_p * phi

    mask = sigma < eps_sigma
    EI[mask] = 0.0
    dEI[mask] = 0.0
    return EI, dEI, mu, sigma


# ------------------------------------------------------------
# Find global maximizer of EI on [lo,hi] via derivative roots
# (roots where dEI changes + -> - are local maxima)
# ------------------------------------------------------------
def maximize_ei_via_roots_1d(gp, lo, hi, f_best, grid_n=1200, tol=1e-10):
    xg = np.linspace(lo, hi, grid_n)
    EI_g, dEI_g, *_ = ei_min_and_derivative_1d(gp, xg, f_best)

    # Candidate points: boundaries always
    candidates = [lo, hi]

    # Find sign changes + -> - in dEI (local maxima)
    for i in range(grid_n - 1):
        a, b = xg[i], xg[i + 1]
        da, db = dEI_g[i], dEI_g[i + 1]

        # strict + to - crossing
        if da > 0 and db < 0:
            def dEI_scalar(x):
                return float(ei_min_and_derivative_1d(gp, np.array([x]), f_best)[1][0])

            # brentq needs opposite signs (we have that)
            try:
                x_root = brentq(dEI_scalar, a, b, xtol=1e-12, rtol=1e-10, maxiter=200)
                candidates.append(x_root)
            except ValueError:
                pass

        # also accept near-zero derivative grid points (flat-ish top)
        if abs(da) < tol:
            candidates.append(a)

    # Evaluate EI at candidates and choose the best
    candidates = np.array(sorted(set(candidates)))
    EI_cand = ei_min_and_derivative_1d(gp, candidates, f_best)[0]
    j = int(np.argmax(EI_cand))
    x_star = float(candidates[j])
    ei_star = float(EI_cand[j])

    # derivative at the chosen x (for sanity)
    dEI_star = float(ei_min_and_derivative_1d(gp, np.array([x_star]), f_best)[1][0])
    return x_star, ei_star, dEI_star


# -------------------------
# Make 5 iteration plots
# -------------------------
def make_5_bo_plots(bounds=(0.0, 1.0), n_init=4, n_iters=5, seed=2):
    lo, hi = bounds
    rng = np.random.default_rng(seed)

    # Initial samples
    X = lo + (hi - lo) * rng.random(n_init)
    X.sort()
    y = f_true(X)

    # GP model (RBF)
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=0.2, length_scale_bounds=(1e-2, 1e2))
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,            # obs noise variance (set larger if noisy)
        normalize_y=False,     # IMPORTANT: keep EI on same scale as y
        n_restarts_optimizer=5,
        random_state=seed
    )

    xgrid = np.linspace(lo, hi, 1500)
    ytrue_grid = f_true(xgrid)

    for t in range(1, n_iters + 1):
        gp.fit(X.reshape(-1, 1), y.reshape(-1))
        f_best = float(np.min(y))

        EI, dEI, mu, sig = ei_min_and_derivative_1d(gp, xgrid, f_best)

        # Choose next point by stationary points of EI
        x_next, ei_next, dEI_next = maximize_ei_via_roots_1d(gp, lo, hi, f_best)

        y_next = float(f_true(x_next))

        # ---- Plot ----
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.8), sharex=True)

        # Top: true f, GP mean +/- 2σ, samples, next
        ax1.plot(xgrid, ytrue_grid, linewidth=2, label="true f(x)")
        ax1.plot(xgrid, mu, linewidth=2, label="GP mean")
        ax1.fill_between(xgrid, mu - 2 * sig, mu + 2 * sig, alpha=0.25, label="GP ± 2σ")
        ax1.scatter(X, y, s=55, edgecolor="black", label="samples", zorder=3)
        ax1.scatter([x_next], [y_next], marker="*", s=240, edgecolor="black",
                    label="next (max EI)", zorder=4)
        ax1.set_ylabel("f(x)  (minimize)")
        ax1.set_title(
            f"Iteration {t}/5 | f_best={f_best:.4f} | max EI≈{ei_next:.3e} | dEI(x_next)≈{dEI_next:.2e}"
        )
        ax1.legend(loc="upper right", framealpha=0.9)

        # Bottom: EI and dEI/dx (same axis, no twin axis = easier to read zero crossing)
        ax2.plot(xgrid, EI, linewidth=2, label="EI(x)")
        ax2.plot(xgrid, dEI, linestyle="--", linewidth=2, label="dEI/dx")
        ax2.axhline(0.0, linewidth=1)
        ax2.axvline(x_next, linewidth=1.2)
        ax2.scatter([x_next], [np.interp(x_next, xgrid, EI)], marker="*", s=170,
                    edgecolor="black", zorder=4)

        ax2.set_xlabel("x")
        ax2.set_ylabel("EI and its derivative")
        ax2.legend(loc="upper right", framealpha=0.9)

        plt.tight_layout()
        out = f"bo_iter_{t}.png"
        plt.savefig(out, dpi=160)
        plt.close(fig)
        print("Saved:", out)

        # BO update
        X = np.append(X, x_next)
        y = np.append(y, y_next)
        order = np.argsort(X)
        X, y = X[order], y[order]


In [11]:
make_5_bo_plots()

Saved: bo_iter_1.png
Saved: bo_iter_2.png
Saved: bo_iter_3.png
Saved: bo_iter_4.png
Saved: bo_iter_5.png
